<a href="https://colab.research.google.com/github/Learn-AIMLOps/Instruction_fine_tunning/blob/main/Final_IFT_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importing Google Colab Secrets
from google.colab import userdata

In [2]:
# Load HuggingFace Token (Colab Secret)
hf_write_token = userdata.get("HF_WRITE_TOKEN")

In [3]:
# Huggingface Login
from huggingface_hub import login
login(token=hf_write_token)

In [7]:
# STEP 0 — Install Everything (Colab)
!pip install -q transformers datasets peft bitsandbytes accelerate trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 36.1 MB/s eta 0:00:00


In [8]:
# STEP 1 — Load Base + Domain Adapter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "Qwen/Qwen2-1.5B"
domain_adapter_name = "RohitGu/Qwen2-1.5B-LLMOps-LoRA"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, domain_adapter_name)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

In [9]:
# STEP 2 — Merge Domain Adapter (VERY IMPORTANT)
model = model.merge_and_unload()

model.save_pretrained("merged_domain_model")
tokenizer.save_pretrained("merged_domain_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('merged_domain_model/tokenizer_config.json',
 'merged_domain_model/chat_template.jinja',
 'merged_domain_model/tokenizer.json')

In [10]:
# STEP 3 — Test Pre-IFT Model (Domain Only)
model.eval()

prompt = """### Instruction:
Explain the LLMOps workflow.

### Input:
Based on domain guide.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    repetition_penalty=1.2,
    do_sample=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


### Instruction:
Explain the LLMOps workflow.

### Input:
Based on domain guide.

### Response:
LLMOps is a platform that provides an end-to-end solution for building and deploying large language models (LLMs). It consists of several components, including:

1. Model training: LLMOps uses pre-trained or custom-trained models to generate text.
2. Data generation: The system generates data based on user input using natural language processing techniques such as tokenization, stemming, and part-of-speech tagging.
3. Text summarization: LLMOps can summarize long documents by extracting key information from them.
4. Chatbot development: LLMOps allows developers to build chatbots with advanced capabilities like sentiment analysis, question answering, and knowledge base integration.
5. Natural Language Generation: LLMOps enables users to create personalized content through generating new texts in various formats such as emails, social media posts, blog articles, etc.,


In [11]:
# STEP 4 — Reload Merged Model in 4-bit for IFT

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normalized Float 4 (usually better than fp4)
    bnb_4bit_compute_dtype=torch.float16, # Keeps computation fast
    bnb_4bit_use_double_quant=True       # Saves a bit more VRAM
)

# Load the model with the config
model = AutoModelForCausalLM.from_pretrained(
    "merged_domain_model",
    quantization_config=bnb_config,      # Pass the config here
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [12]:
# STEP 5 — Apply NEW LoRA (For IFT)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [13]:
# STEP 6 — Load & Format IFT Dataset

# Sub Step 1 --
# IFT Data cleanliness (clears bad formating or cleansing the datasets)
import json
from datasets import Dataset

def load_robust_jsonl(file_path):
    data = []
    skipped = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_number, line in enumerate(f, start=1):
            try:
                entry = json.loads(line)

                # Ensure required keys exist
                instruction = entry.get("instruction", "").strip()
                input_text = entry.get("input", "").strip()
                response = entry.get("response", "")

                # Skip if instruction or response missing
                if not instruction or not response:
                    skipped += 1
                    continue

                # If response is dict/list → convert to string
                if not isinstance(response, str):
                    response = json.dumps(response, ensure_ascii=False)

                # Final cleaned entry
                cleaned = {
                    "instruction": instruction,
                    "input": input_text,
                    "response": response.strip()
                }

                data.append(cleaned)

            except Exception as e:
                skipped += 1
                continue

    print(f"Loaded {len(data)} valid samples.")
    print(f"Skipped {skipped} malformed samples.")

    return Dataset.from_list(data)

# Sub Step 2 -- Load Your Dataset
raw_dataset = load_robust_jsonl("/content/ift_dataset.jsonl")


# Sub Step3 -- Format for Training
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['response']}"""
    }

dataset = raw_dataset.map(format_example)
print(f"Final dataset size: {len(dataset)}")

Loaded 91 valid samples.
Skipped 9 malformed samples.


Map:   0%|          | 0/91 [00:00<?, ? examples/s]

Final dataset size: 91


In [14]:
import trl
print(trl.__version__)

0.29.0


In [18]:
# Quick Sanity Check
print(dataset.column_names)
print(dataset[0])

['instruction', 'input', 'response', 'text']
{'instruction': 'Explain why small language models offer a more resource-efficient alternative than larger ones.', 'input': 'Small language models are often used in conjunction with large-scale pre-trained models to scale up the computational resources required to train them.', 'response': 'Small language models require less computational power and data to train compared to larger models, making them a more resource-efficient option.', 'text': '### Instruction:\nExplain why small language models offer a more resource-efficient alternative than larger ones.\n\n### Input:\nSmall language models are often used in conjunction with large-scale pre-trained models to scale up the computational resources required to train them.\n\n### Response:\nSmall language models require less computational power and data to train compared to larger models, making them a more resource-efficient option.'}


In [24]:
# STEP 7 — Train (QLoRA SFT)
from trl import SFTTrainer, SFTConfig

# 🔹 Must be set BEFORE trainer
tokenizer.model_max_length = 1024
tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False

sft_config = SFTConfig(
    output_dir="./instruction_adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,          # ✅ REQUIRED
    processing_class=tokenizer,     # ✅ This replaces tokenizer=
    args=sft_config,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Step,Training Loss
10,2.052946
20,1.936160
30,1.829869


TrainOutput(global_step=36, training_loss=1.933670441309611, metrics={'train_runtime': 222.5837, 'train_samples_per_second': 1.227, 'train_steps_per_second': 0.162, 'total_flos': 225882011888640.0, 'train_loss': 1.933670441309611})

In [25]:
# STEP 8 — Save Instruction Adapter
model.save_pretrained("instruction_adapter")
tokenizer.save_pretrained("instruction_adapter")

('instruction_adapter/tokenizer_config.json',
 'instruction_adapter/chat_template.jinja',
 'instruction_adapter/tokenizer.json')

In [27]:
# STEP 9 — Push Instruction Adapter to Hugging Face
model.push_to_hub("RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA", token=hf_write_token)
tokenizer.push_to_hub("RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA", token=hf_write_token)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  14%|#4        |  626kB / 4.37MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9r15asdi/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA/commit/a554c5f6b3a97b43484ada237c8824e00acd867e', commit_message='Upload tokenizer', commit_description='', oid='a554c5f6b3a97b43484ada237c8824e00acd867e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA'), pr_revision=None, pr_num=None)

In [29]:
# STEP 10 — Test FINAL Model (Post-IFT)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 🔹 Define 4-bit config properly
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 🔹 Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "merged_domain_model",
    quantization_config=bnb_config,   # ✅ correct way
    device_map="auto",
)

from peft import PeftModel

# 🔹 Load LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    "RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA"
)

model.eval()

# 🔹 Generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    repetition_penalty=1.2
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/973 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


### Instruction:
Explain the LLMOps workflow.

### Input:
Based on domain guide.

### Response:
The LLMOps workflow involves several steps, including data collection and preprocessing, model training and evaluation, deployment, monitoring, and maintenance. It also includes tools for tracking performance metrics such as accuracy, precision, recall, F1 score, etc., to ensure that models are performing well in real-world scenarios. Additionally, it requires continuous updates based on feedback from users or stakeholders to maintain relevance over time.
